# 6 Scaling: From One Document to a Corpus

In Sections 3 and 4, we built a complete pipeline: ingestion with Docling, word-boundary-respecting chunking, embedding with Granite, retrieval from ChromaDB, and grounded generation. In Section 5, we evaluated the results and proved that retrieval adds real, measurable value.

All of that was built against a single document. One rulebook. One PDF.

No customer has one document.

They have twelve. Or two hundred. Or twelve thousand. They have policy manuals that reference procedure guides that reference training materials that contradict last year's version of themselves. They have documents written by different people in different decades using different terminology for the same concepts.

The question we need to answer now is not whether the pipeline works. We proved that. The question is whether it survives.


## 6.1 The System, Not the Step
Up to this point, we have worked through a progression that felt sequential: ingest a document, chunk it, embed it, retrieve from it, evaluate the results. Each step had a purpose. Each step taught something.

But here is the shift that matters: those were not steps. They were pipeline stages.

The distinction is important. A step is something you do once. A pipeline is something that runs again, against new documents, against updated documents, against documents that did not exist when you first built the system. The customer's corpus is not static. Their PDFs change. New policies get published. Old manuals get revised. The model might get swapped out. The embedding strategy might need tuning.

If the system cannot survive any of those changes, it is not a system. It is a demo that worked once.

The answer, and this is the key insight for the field: the pipeline does not change. The input does.

The same Docling conversion, the same word-boundary-respecting chunking, the same Granite embedding model, the same ChromaDB vector store, the same retrieval function. All of it carries forward unchanged. What changes is the loop: instead of processing one PDF, we iterate over a directory of them.

That is not a trivial observation. It means the architectural decisions we made in Sections 3 and 4 (the chunking strategy, the overlap size, the embedding model) are now load-bearing. They apply to every document in the corpus. If they were wrong for one document, they will be wrong for all of them.

>Facilitator note: This is where you land the reframe. Ingestion was not a task. Chunking was not a tweak. Evaluation was not an afterthought. They were all pipeline stages. And now we see them working together.

## 6.2 Making the Workflow Explicit

Now we name the system. Walk through the end-to-end flow in plain language before running any code:

1. Source documents enter the system (PDFs in a directory)
1. Each document is parsed and normalized (Docling converts to Markdown)
1. Content is structured and chunked (word-boundary-respecting splits with overlap)
1. Chunks are embedded (Granite embedding model, loaded once, reused for all documents)
1. Everything is loaded into a vector store (ChromaDB, with metadata tracking provenance)
1. The model is exercised against the corpus (RAG queries with source attribution)
1. Outputs are evaluated (same question set from Section 5, re-run against the new collection)
1. Decisions are made about what, if anything, needs to change

Nothing here is exotic. Every stage is observable. Every stage is replaceable. If the chunking strategy is not working, you swap it. If the embedding model is not capturing domain nuance, you try another. If Docling mishandles a particular document layout, you investigate and fix the ingestion. The pipeline gives you a place to look when something goes wrong.

>Field takeaway: If you cannot point to where something went wrong, you do not have a pipeline. You have a demo.

The code that follows is deliberately familiar. The helper functions are identical to Section 4. The Docling configuration is the same. The embedding model is the same. What is new is the orchestration: a loop that processes every PDF in a directory and accumulates the results into a single, searchable collection. Each chunk carries metadata that records which document it came from. That single design decision is what makes source attribution possible downstream.

### 6.2.1 Setup MaaS Connection
Setup code to connect to MaaS endpoint.

In [1]:
import sys
sys.path.insert(0, "..")
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base
import requests
import json
import numpy as np
import os
os.environ["TQDM_DISABLE"] = "1"

__import__('pysqlite3')
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

import chromadb
from pathlib import Path
from sentence_transformers import SentenceTransformer
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption
url_chat = f"{endpoint_base}/chat/completions"
model_id = "granite-3-2-8b-instruct"
print("\u2705 All imports loaded")

✅ All imports loaded


### 6.2.2 Helper functions
These are the same `chunk_text_markdown` and `add_to_collection` functions from Section 4. Unchanged. The pipeline is the same; we are just feeding it more documents.



In [2]:
def chunk_text_markdown(text, chunk_size=1000, overlap=200):
    import re
    
    # Split on markdown headings (keep the heading with the text that follows)
    # and on double newlines (paragraph boundaries)
    sections = re.split(r'(?=\n#{1,6} )', text)
    
    # Further split each section on paragraph breaks (double newline)
    blocks = []
    for section in sections:
        paragraphs = re.split(r'\n{2,}', section)
        for p in paragraphs:
            stripped = p.strip()
            if stripped:
                blocks.append(stripped)
    
    # Accumulate blocks into chunks
    chunks = []
    current_chunk = ""
    
    for block in blocks:
        # If a single block exceeds chunk_size, split it at word boundaries
        if len(block) > chunk_size:
            # Flush current chunk first
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
                current_chunk = ""
            
            # Word-boundary fallback for oversized blocks
            start = 0
            while start < len(block):
                end = start + chunk_size
                if end < len(block):
                    space_pos = block.rfind(" ", start, end)
                    if space_pos > start:
                        end = space_pos
                sub = block[start:end].strip()
                if sub:
                    chunks.append(sub)
                next_start = end - overlap
                start = max(next_start, start + 1)
            continue
        
        # Would adding this block exceed the budget?
        candidate = (current_chunk + "\n\n" + block).strip() if current_chunk else block
        if len(candidate) > chunk_size and current_chunk.strip():
            chunks.append(current_chunk.strip())
            current_chunk = block
        else:
            current_chunk = candidate
    
    # Flush the last chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    
    # Apply overlap: prepend tail of previous chunk to each subsequent chunk
    if overlap > 0 and len(chunks) > 1:
        overlapped = [chunks[0]]
        for i in range(1, len(chunks)):
            prev = chunks[i - 1]
            # Take the last `overlap` characters, back up to word boundary
            if len(prev) > overlap:
                tail_start = len(prev) - overlap
                space_pos = prev.find(" ", tail_start)
                if space_pos != -1:
                    tail = prev[space_pos:].strip()
                else:
                    tail = prev[tail_start:].strip()
            else:
                tail = prev.strip()
            overlapped.append((tail + "\n\n" + chunks[i]).strip())
        chunks = overlapped
    
    return chunks


In [3]:
def add_to_collection(collection, chunks, embeddings, metadatas=None, ids=None, batch_size=5000):
    """Batch-load chunks into a ChromaDB collection."""
    for i in range(0, len(chunks), batch_size):
        end = min(i + batch_size, len(chunks))
        batch_kwargs = {
            "documents": chunks[i:end],
            "embeddings": embeddings[i:end].tolist() if hasattr(embeddings, 'tolist') else embeddings[i:end],
            "ids": ids[i:end] if ids else [f"chunk_{j}" for j in range(i, end)],
        }
        if metadatas:
            batch_kwargs["metadatas"] = metadatas[i:end]
        collection.add(**batch_kwargs)
    print(f"\u2705 Loaded {collection.count()} chunks into the vector store")

### 6.2.3 Discover the Corpus
The pipeline scans the `docs/` directory for every PDF. If you have placed additional documents there, they will be picked up automatically. The pipeline is input-agnostic.


In [4]:
docs_dir = Path("../docs")
pdf_files = sorted([
    f for f in docs_dir.rglob("*.pdf")
    if ".ipynb_checkpoints" not in str(f)
])

print(f"Found {len(pdf_files)} PDFs:")
for f in pdf_files:
    print(f"  {f.relative_to(docs_dir)}")


Found 6 PDFs:
  3rdEdition/Basic-Fantasy-RPG-Rules-r107-bookmarked.pdf
  3rdEdition/Basic-Fantasy-RPG-Rules-r107-lite.pdf
  3rdEdition/Basic-Fantasy-RPG-Rules-r107.pdf
  BFRPG-Monster-Index-r7.pdf
  Basic-Fantasy-RPG-Rules-r142.pdf
  EE1-Equipment-Emporium-r33.pdf


> Note: I added the checkpoint filter. That .ipynb_checkpoints PDF in your current output would cause garbage during conversion.

### 6.2.4 Load the Embedding Model
We load the embedding model once and reuse it for all documents. Same Granite Embedding model from Section 4.


In [5]:
print("Loading Granite embedding model...")
embed_model = SentenceTransformer("ibm-granite/granite-embedding-30m-english")
print("\u2705 Embedding model loaded")

Loading Granite embedding model...


/opt/app-root/lib64/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


✅ Embedding model loaded


### 6.2.5 Configure Docling Converter
Same configuration as Section 3. OCR is disabled because all documents in this lab contain extractable text. In a production deployment with scanned documents, you would enable OCR here and the rest of the pipeline would not change.


In [6]:
pdf_options = PdfPipelineOptions()
pdf_options.do_ocr = False

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)
    }
)
print("\u2705 Docling converter configured (OCR disabled)")

✅ Docling converter configured (OCR disabled)


## 6.3 Run the Pipeline
This is the same sequence from Sections 3 and 4, inside a loop. For each PDF: convert, chunk, embed, and accumulate. Every chunk carries metadata with its source filename, and every chunk ID encodes its origin document. That is not decorative. It is what makes source attribution possible at query time.


>Facilitator note: This cell will take about ten minutes depending on how many PDFs are in the directory. Watch the progress output. If it stalls on a particular document for more than three minutes, note which file caused the delay and move to pre-built outputs. The learning objective does not require every PDF to finish processing.


In [7]:
import time

from chromadb.errors import NotFoundError
client = chromadb.PersistentClient(path="../chroma_db")

# Delete if exists (safe for re-runs)
try:
    client.delete_collection("rpg_rules_multi")
except (ValueError, NotFoundError):
    pass

collection_multi = client.create_collection(
    name="rpg_rules_multi",
    metadata={"hnsw:space": "cosine"}
)

pipeline_start = time.time()
skipped = []

for pdf_idx, pdf_path in enumerate(pdf_files, 1):
    filename = pdf_path.name
    print(f"\n{'='*60}")
    print(f"[{pdf_idx}/{len(pdf_files)}] Processing: {filename}")
    print(f"{'='*60}")

    doc_start = time.time()

    try:
        # Step 1: Convert PDF to Markdown
        print("  Converting PDF to markdown...")
        step_start = time.time()
        result = converter.convert(str(pdf_path))
        markdown_text = result.document.export_to_markdown()
        print(f"  Extracted {len(markdown_text):,} characters ({time.time() - step_start:.1f}s)")
    except Exception as e:
        print(f"  ⚠️  Skipping {filename}: {e}")
        skipped.append(filename)
        continue

    # Step 2: Chunk
    step_start = time.time()
    chunks = chunk_text_markdown(markdown_text, chunk_size=1000, overlap=200)
    print(f"  Created {len(chunks)} chunks ({time.time() - step_start:.1f}s)")

    # Step 3: Embed
    print("  Embedding chunks...")
    step_start = time.time()
    embeddings = embed_model.encode(chunks, show_progress_bar=False, batch_size=64)
    print(f"  Embedded {len(chunks)} chunks ({time.time() - step_start:.1f}s)")

    # Step 4: Load directly into ChromaDB
    step_start = time.time()
    metadatas = [{"source": filename} for _ in chunks]
    ids = [f"{filename}_chunk_{i}" for i in range(len(chunks))]

    add_to_collection(
        collection_multi,
        chunks,
        embeddings,
        metadatas=metadatas,
        ids=ids
    )
    print(f"  Loaded into ChromaDB ({time.time() - step_start:.1f}s)")

    doc_elapsed = time.time() - doc_start
    print(f"  ✅ Done in {doc_elapsed:.1f}s — collection now has {collection_multi.count()} chunks")

pipeline_elapsed = time.time() - pipeline_start
print(f"\n{'='*60}")
print(f"Pipeline complete: {collection_multi.count()} total chunks from {len(pdf_files) - len(skipped)} of {len(pdf_files)} document(s)")
print(f"Total pipeline time: {pipeline_elapsed:.1f}s ({pipeline_elapsed/60:.1f} min)")
if skipped:
    print(f"Skipped {len(skipped)} file(s): {', '.join(skipped)}")
print(f"{'='*60}")


[1/6] Processing: Basic-Fantasy-RPG-Rules-r107-bookmarked.pdf
  Converting PDF to markdown...
  Extracted 756,799 characters (610.6s)
  Created 995 chunks (0.0s)
  Embedding chunks...
  Embedded 995 chunks (65.6s)
✅ Loaded 995 chunks into the vector store
  Loaded into ChromaDB (0.7s)
  ✅ Done in 676.9s — collection now has 995 chunks

[2/6] Processing: Basic-Fantasy-RPG-Rules-r107-lite.pdf
  Converting PDF to markdown...
  Extracted 756,587 characters (600.5s)
  Created 995 chunks (0.0s)
  Embedding chunks...
  Embedded 995 chunks (66.5s)
✅ Loaded 1990 chunks into the vector store
  Loaded into ChromaDB (0.7s)
  ✅ Done in 667.8s — collection now has 1990 chunks

[3/6] Processing: Basic-Fantasy-RPG-Rules-r107.pdf
  Converting PDF to markdown...
  Extracted 756,799 characters (608.2s)
  Created 995 chunks (0.0s)
  Embedding chunks...
  Embedded 995 chunks (66.7s)
✅ Loaded 2985 chunks into the vector store
  Loaded into ChromaDB (0.7s)
  ✅ Done in 675.6s — collection now has 2985 chunks

Six documents. Nearly 5,000 chunks. Just under 9 minutes on a single GPU. That's the number to sit with for a moment, because your customer won't have six documents.

If the ratio holds roughly linear, a hundred documents at this complexity would take about two and a half hours. A thousand documents would take a full day. And that assumes everything goes cleanly, no OCR, no corrupted files, no unexpected formatting that stalls the parser.
This is where the conversation shifts from 'does the pipeline work' to 'how do we run the pipeline.' That's an infrastructure question, not an AI question. And it's one of the most important conversations you'll have with a customer, because the answer shapes cost, timeline, and operational ownership.

The levers are real. Parallelism across multiple GPUs cuts wall-clock time dramatically. Batch scheduling means ingestion can run overnight or over a weekend without tying up interactive resources. Incremental pipelines mean you only re-process documents that changed, not the entire corpus every time. Caching converted markdown means you never pay the conversion cost twice for the same file.

None of that is exotic. It's the same kind of operational planning that any data pipeline requires. But it needs to happen early, not after someone tries to ingest ten thousand PDFs from a laptop and wonders why it took a week.

The question to bring into a customer meeting isn't 'can we process your documents.' We just proved we can. The question is 'how often do your documents change, how many are there, and what does your timeline look like.' Those answers drive architecture decisions that have nothing to do with models and everything to do with whether the system survives first contact with production.

## 6.4 Retrieval with Source Attribution

In a single-document pipeline, you always know where an answer came from. There's only one source. That convenience disappears the moment you add a second document.

With multiple documents, the customer will always ask: "Where did that come from?"

This is not a nice-to-have. It is the difference between a demo and a deployable system. In regulated industries, in legal contexts, in any environment where someone has to explain a decision to a stakeholder, source attribution is mandatory.

The notebook introduces a ask_with_rag_sources function that extends the RAG query from Section 4. It returns not just the model's answer, but the source document for each retrieved chunk. When you run the test question "What happens if a Thief fails an Open Locks attempt?" you don't just get the correct answer. You get a list of which documents contributed to that answer and which specific chunks were retrieved.

This is where metadata pays off. Every chunk we loaded carried a source field in its metadata. That single design decision, made during ingestion, is what enables traceability at query time. If we'd skipped it, we'd have answers with no provenance. Accurate answers, perhaps, but unverifiable ones. And unverifiable answers don't survive contact with a compliance team.

Key point for the field:

>Source attribution isn't a feature you add later. It's a design decision you make at ingestion. If the metadata isn't there when chunks are created, it's not there when answers are generated.

Now, let's add the function `ask_with_rag_sources`

In [8]:
def ask_with_rag_sources(question, collection, embed_model, n_results=3):
    # Retrieve with metadata
    query_embedding = embed_model.encode(question).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas"]
    )

    chunks = results["documents"][0]
    metadatas = results["metadatas"][0]
    sources = [(chunk, meta.get("source", "unknown")) for chunk, meta in zip(chunks, metadatas)]

    # Build grounded prompt
    context = "\n\n---\n\n".join(chunks)
    system_msg = (
        "Answer the question using only the provided context. "
        "If the context does not contain enough information, say so."
    )
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model_id,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        "max_tokens": 300,
        "temperature": 0
    }

    response = requests.post(url_chat, headers=headers, json=data)
    response.raise_for_status()
    answer = response.json()["choices"][0]["message"]["content"]

    return answer, sources

The function queries ChromaDB with the embedded question, retrieves the top matching chunks along with their metadata, builds a grounded prompt, and returns both the answer and the list of sources. Nothing here is new individually. The retrieval is the same as Section 4. The prompt structure is the same. 

What's new is that every answer now carries provenance. Run the next cell to see it in action.

In [9]:
test_question = "What happens if a Thief fails an Open Locks attempt?"
answer, sources = ask_with_rag_sources(test_question, collection_multi, embed_model)

print(f"Question: {test_question}\n")
print(f"Answer:\n{answer}\n")
print("=" * 60)
print("SOURCE ATTRIBUTION")
print("=" * 60)
for i, (chunk_text_preview, source) in enumerate(sources, 1):
    print(f"\n--- Chunk {i} | Source: {source} ---")
    print(chunk_text_preview[:200] + "...")

Question: What happens if a Thief fails an Open Locks attempt?

Answer:
If a Thief fails an Open Locks attempt, they must wait until they have gained another level of experience before trying again.

SOURCE ATTRIBUTION

--- Chunk 1 | Source: Basic-Fantasy-RPG-Rules-r107-bookmarked.pdf ---
82 |             98 |              91 |            98 |     72 |       92 |
|            20 |           88 |             83 |             99 |              93 |            99 |     73 |       95 |

Op...

--- Chunk 2 | Source: Basic-Fantasy-RPG-Rules-r107-lite.pdf ---
82 |             98 |              91 |            98 |     72 |       92 |
|            20 |           88 |             83 |             99 |              93 |            99 |     73 |       95 |

Op...

--- Chunk 3 | Source: Basic-Fantasy-RPG-Rules-r107.pdf ---
82 |             98 |              91 |            98 |     72 |       92 |
|            20 |           88 |             83 |             99 |              93 |            99 | 

Look at what just happened. The answer is correct, but the retrieval is wasteful. All three chunks came from three different versions of the same third-edition rulebook. The current edition (r142) was not retrieved at all. This is the redundancy problem in practice: near-duplicate documents consume retrieval slots that could have gone to supplementary material like the Monster Index or Equipment Emporium.
In a customer environment with hundreds of versioned documents, this problem gets worse, not better. The fix is not a model change. It is corpus curation: deduplicating versions, marking authoritative sources, or filtering retrieval by metadata. The pipeline gives you the visibility to see the problem. The solution is upstream.

> **Facilitator note:** Corpus curation, including deduplication, version management, and metadata-based filtering, is a natural next step. It is outside the scope of this lab but is often the highest-value work in a real engagement.

## 6.5 Evaluation: Same Questions, Multi-Document Corpus

We re-run the same five evaluation questions from Section 5 against the multi-document collection. Same questions. Same reference answers. Same evaluation structure.

This is intentional. If we changed the questions, we couldn't isolate what changed. By holding the evaluation constant, we can see exactly what the multi-document pipeline introduced: did answers improve? Did they regress? Did source attribution land on the right documents?
What participants should watch for:

* **Redundancy.** You just saw this. Three retrieval slots consumed by three versions of the same document. At scale, this gets worse. When both a rulebook and a supplement describe how initiative works, retrieval may return near-duplicate chunks, wasting slots on repetition rather than breadth.

* **Noise.** More documents means more chunks competing for the top retrieval positions. Irrelevant chunks from unrelated documents can displace relevant ones. A chunk about character creation might edge out the one about combat initiative if the embedding similarity scores are close enough.

* **Conflict.** Different documents may state different versions of the same rule. This is extremely common in enterprise settings: policy documents that were revised but never retired, training manuals that reference outdated procedures, regional variations that contradict the corporate standard. The model doesn't know which version is authoritative. It works with whatever retrieval gives it.

None of these are model problems. They are corpus curation problems. And that distinction matters enormously for how the field frames the conversation with customers.

After running the comparison between the single-document collection (from Section 4) and the multi-document collection, participants will see the differences firsthand. Sometimes the multi-document pipeline produces better answers because it has access to supplementary material. Sometimes it produces noisier answers because irrelevant documents compete for attention.

The evaluation framework from Section 5 is what makes these differences visible. Without it, you'd be guessing. With it, you can point to exactly where the system improved and where it regressed.
Field takeaway:

>Field takeaway: If adding a new document makes existing answers worse, you need to know immediately. That's not a model failure. It's a signal that your corpus needs curation.

Let's define the questions.

In [10]:
eval_questions = [
    {
        "id": "q1",
        "question": "What happens if a Thief fails an Open Locks attempt?",
        "reference_answer": "The Thief must wait until they have gained another level of experience before trying again. It may only be tried once per lock."
    },
    {
        "id": "q2",
        "question": "Why can't Elves roll higher than a d6 for hit points?",
        "reference_answer": "Elves use a d6 for hit dice because they are a combination Fighter/Magic-User class, and their hit die reflects the Magic-User side."
    },
    {
        "id": "q3",
        "question": "Can a Fighter use magic-user scrolls?",
        "reference_answer": "No. Only spell casters can use magic-user scrolls."
    },
    {
        "id": "q4",
        "question": "How is initiative determined in combat?",
        "reference_answer": "Each side rolls 1d6 at the beginning of each round. The side with the highest roll acts first."
    },
    {
        "id": "q5",
        "question": "What is the maximum number of retainers a character can hire?",
        "reference_answer": "The number of retainers is based on the character's Charisma bonus. A character with average Charisma can hire up to 4."
    },
]

print(f"Evaluation set: {len(eval_questions)} questions")
for q in eval_questions:
    print(f"  [{q['id']}] {q['question']}")

Evaluation set: 5 questions
  [q1] What happens if a Thief fails an Open Locks attempt?
  [q2] Why can't Elves roll higher than a d6 for hit points?
  [q3] Can a Fighter use magic-user scrolls?
  [q4] How is initiative determined in combat?
  [q5] What is the maximum number of retainers a character can hire?


With the evaluation questions defined and the multi-document collection loaded, we run the same questions against the expanded corpus. Watch the source attribution in the results. Which documents are the answers drawing from? Are they pulling from the current edition, the older versions, or the supplements?

In [11]:
eval_results = []

print("Running evaluation against multi-document collection...\n")
for q in eval_questions:
    print(f"[{q['id']}] {q['question']}")
    answer, sources = ask_with_rag_sources(q["question"], collection_multi, embed_model)
    eval_results.append({
        "id": q["id"],
        "question": q["question"],
        "reference_answer": q["reference_answer"],
        "answer": answer,
        "sources": sources
    })
    print(f"  \u2705 Done\n")

print(f"All {len(eval_results)} questions complete.")

Running evaluation against multi-document collection...

[q1] What happens if a Thief fails an Open Locks attempt?
  ✅ Done

[q2] Why can't Elves roll higher than a d6 for hit points?
  ✅ Done

[q3] Can a Fighter use magic-user scrolls?
  ✅ Done

[q4] How is initiative determined in combat?
  ✅ Done

[q5] What is the maximum number of retainers a character can hire?
  ✅ Done

All 5 questions complete.


Now we lay out the full results. For each question, you'll see the reference answer, the model's RAG answer, which source documents were used, and a preview of the retrieved chunks. Read this carefully. Compare the sources across questions. Are the same documents dominating retrieval? Are the supplements contributing, or are the older editions crowding them out?

In [12]:
print("=" * 80)
print("MULTI-DOCUMENT EVALUATION RESULTS")
print("=" * 80)

for r in eval_results:
    print(f"\n{'\u2500' * 80}")
    print(f"{r['id'].upper()}: {r['question']}")
    print(f"{'\u2500' * 80}")

    print(f"\n\U0001f4d6 Reference:\n   {r['reference_answer']}")
    print(f"\n\U0001f4da RAG Answer (multi-doc):\n   {r['answer']}")

    print(f"\n   Sources:")
    source_files = set(src for _, src in r["sources"])
    for src in source_files:
        print(f"     - {src}")

    print(f"\n   Retrieved chunks:")
    for i, (chunk_preview, source) in enumerate(r["sources"], 1):
        print(f"     [{i}] ({source}) {chunk_preview[:120]}...")

MULTI-DOCUMENT EVALUATION RESULTS

────────────────────────────────────────────────────────────────────────────────
Q1: What happens if a Thief fails an Open Locks attempt?
────────────────────────────────────────────────────────────────────────────────

📖 Reference:
   The Thief must wait until they have gained another level of experience before trying again. It may only be tried once per lock.

📚 RAG Answer (multi-doc):
   If a Thief fails an Open Locks attempt, they must wait until they have gained another level of experience before trying again.

   Sources:
     - Basic-Fantasy-RPG-Rules-r107-bookmarked.pdf
     - Basic-Fantasy-RPG-Rules-r107-lite.pdf
     - Basic-Fantasy-RPG-Rules-r107.pdf

   Retrieved chunks:
     [1] (Basic-Fantasy-RPG-Rules-r107-bookmarked.pdf) 82 |             98 |              91 |            98 |     72 |       92 |
|            20 |           88 |            ...
     [2] (Basic-Fantasy-RPG-Rules-r107-lite.pdf) 82 |             98 |              91 |      

The raw output works for debugging, but it's hard to scan quickly. The next cell renders the same results in a formatted view that makes it easier to compare reference answers against RAG answers, spot which source documents are contributing, and see what the retrieved chunks actually contain.

In [13]:
from IPython.display import display, HTML

html_parts = []
html_parts.append("""
<style>
.eval-container {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    max-width: 900px;
}
.eval-header {
    font-size: 1.4em;
    font-weight: 700;
    padding: 12px 16px;
    margin-bottom: 16px;
    border-bottom: 3px solid #e74c3c;
    color: var(--jp-ui-font-color0, #333);
}
.eval-card {
    border: 1px solid var(--jp-border-color1, #ddd);
    border-radius: 8px;
    margin-bottom: 20px;
    overflow: hidden;
    background: var(--jp-layout-color1, #fff);
}
.eval-card-header {
    padding: 12px 16px;
    font-weight: 600;
    font-size: 1.05em;
    background: var(--jp-layout-color2, #f5f5f5);
    border-bottom: 1px solid var(--jp-border-color1, #ddd);
    color: var(--jp-ui-font-color0, #333);
}
.eval-card-body {
    padding: 16px;
}
.eval-section {
    margin-bottom: 14px;
}
.eval-label {
    font-size: 0.8em;
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 0.05em;
    margin-bottom: 4px;
    color: var(--jp-ui-font-color2, #888);
}
.eval-reference {
    padding: 10px 14px;
    border-radius: 6px;
    background: rgba(52, 152, 219, 0.08);
    border-left: 3px solid #3498db;
    color: var(--jp-ui-font-color0, #333);
}
.eval-answer {
    padding: 10px 14px;
    border-radius: 6px;
    background: rgba(46, 204, 113, 0.08);
    border-left: 3px solid #2ecc71;
    color: var(--jp-ui-font-color0, #333);
}
.eval-sources {
    display: flex;
    flex-wrap: wrap;
    gap: 6px;
    margin-top: 4px;
}
.eval-source-tag {
    font-size: 0.78em;
    padding: 3px 10px;
    border-radius: 12px;
    background: var(--jp-layout-color2, #eee);
    color: var(--jp-ui-font-color1, #555);
    border: 1px solid var(--jp-border-color1, #ddd);
}
.eval-chunk {
    font-size: 0.82em;
    padding: 8px 12px;
    margin-top: 6px;
    border-radius: 4px;
    background: var(--jp-layout-color2, #f8f8f8);
    color: var(--jp-ui-font-color1, #555);
    font-family: 'SFMono-Regular', Consolas, monospace;
    white-space: pre-wrap;
    word-break: break-word;
}
.eval-chunk-label {
    font-size: 0.75em;
    font-weight: 600;
    color: var(--jp-ui-font-color2, #888);
}
</style>
<div class="eval-container">
<div class="eval-header">Multi-Document Evaluation Results</div>
""")

for r in eval_results:
    source_files = set(src for _, src in r["sources"])
    source_tags = "".join(f'<span class="eval-source-tag">{src}</span>' for src in source_files)
    
    chunk_html = ""
    for i, (chunk_preview, source) in enumerate(r["sources"], 1):
        preview = chunk_preview[:150].replace("<", "&lt;").replace(">", "&gt;")
        chunk_html += f"""
        <div class="eval-chunk-label">Chunk {i} / {source}</div>
        <div class="eval-chunk">{preview}...</div>
        """
    
    answer_escaped = r["answer"].replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
    ref_escaped = r["reference_answer"].replace("<", "&lt;").replace(">", "&gt;")
    
    html_parts.append(f"""
    <div class="eval-card">
        <div class="eval-card-header">{r['id'].upper()}: {r['question']}</div>
        <div class="eval-card-body">
            <div class="eval-section">
                <div class="eval-label">Reference Answer</div>
                <div class="eval-reference">{ref_escaped}</div>
            </div>
            <div class="eval-section">
                <div class="eval-label">RAG Answer (Multi-Document)</div>
                <div class="eval-answer">{answer_escaped}</div>
            </div>
            <div class="eval-section">
                <div class="eval-label">Sources</div>
                <div class="eval-sources">{source_tags}</div>
            </div>
            <div class="eval-section">
                <div class="eval-label">Retrieved Chunks</div>
                {chunk_html}
            </div>
        </div>
    </div>
    """)

html_parts.append("</div>")
display(HTML("".join(html_parts)))

Take a moment to read through the results. A few things should stand out.

Q1 and Q4 pulled exclusively from the third-edition documents. The current edition (r142) didn't appear at all. That's the redundancy problem again. Three versions of the same rulebook are crowding out the authoritative source.

Q3 and Q5 show healthier retrieval. The r142 document appears alongside older editions, and the answers are accurate and well-grounded. When retrieval pulls from the right sources, the system works exactly as designed.

Q2 is the most interesting case. The model retrieved relevant chunks from r142 but still couldn't fully answer the question. It acknowledged the limitation honestly instead of hallucinating an explanation. That's actually good behavior. The information needed to answer "why" isn't stated explicitly in any of the documents. It's implicit domain knowledge. This is precisely the kind of failure that would justify investigating model customization later, but only after confirming that no amount of retrieval improvement will surface an answer that doesn't exist in the corpus.

The pattern across all five questions reinforces the same lesson: when the system fails, look upstream before looking at the model. Is the right document being retrieved? Is the right chunk in the collection? Is the information even in the corpus at all? Those questions have to be answered before any conversation about fine-tuning.

## 6.6 Comparison: Single-Document vs Multi-Document
Now we compare the rpg_rules collection from Section 4 (single document) against rpg_rules_multi (full corpus). This is where scaling effects become visible. The same question, asked against two different collections, reveals what changes when you add more documents to the system.

Watch for three things. First, does the answer change? If the single-document answer was correct, does the multi-document answer stay correct or does it drift? Second, do the sources make sense? With more documents available, the retriever has more to choose from. Did it choose well? Third, does the additional material add value, or does it just add noise?

Run the next cell and compare.

In [14]:
# Load the single-doc collection from Section 4
try:
    collection_single = client.get_collection(name="rpg_rules")
    print(f"Single-doc collection: {collection_single.count()} chunks")
    print(f"Multi-doc collection:  {collection_multi.count()} chunks")
except Exception as e:
    print(f"\u26a0\ufe0f  Could not load single-doc collection 'rpg_rules': {e}")
    print("   Run Section 4 first to create it.")
    collection_single = None

if collection_single:
    compare_question = "How is initiative determined in combat?"

    print(f"\n{'=' * 70}")
    print(f"COMPARISON: {compare_question}")
    print(f"{'=' * 70}")

    # Single-doc answer (no source metadata)
    q_emb = embed_model.encode(compare_question).tolist()
    single_results = collection_single.query(
        query_embeddings=[q_emb], n_results=3
    )
    single_context = "\n\n---\n\n".join(single_results["documents"][0])
    headers = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}
    single_data = {
        "model": model_id,
        "messages": [
            {"role": "system", "content": "Answer the question using only the provided context. If the context does not contain enough information, say so."},
            {"role": "user", "content": f"Context:\n{single_context}\n\nQuestion: {compare_question}"}
        ],
        "max_tokens": 300, "temperature": 0
    }
    resp = requests.post(url_chat, headers=headers, json=single_data)
    single_answer = resp.json()["choices"][0]["message"]["content"]

    # Multi-doc answer (with source attribution)
    multi_answer, multi_sources = ask_with_rag_sources(
        compare_question, collection_multi, embed_model
    )

    print(f"\n\U0001f4c4 Single-document RAG ({collection_single.count()} chunks):")
    print(f"   {single_answer}")

    print(f"\n\U0001f4da Multi-document RAG ({collection_multi.count()} chunks):")
    print(f"   {multi_answer}")

    print(f"\n   Sources used:")
    for _, source in multi_sources:
        print(f"     - {source}")

Single-doc collection: 1205 chunks
Multi-doc collection:  4731 chunks

COMPARISON: How is initiative determined in combat?

📄 Single-document RAG (1205 chunks):
   In combat, initiative is determined by rolling 1d6 for each character or monster. This roll is then adjusted by the character's Dexterity bonus. Higher numbers act first. If two characters or monsters have equal numbers, they act simultaneously. The GM may make a single roll for groups of identical monsters at their option.

📚 Multi-document RAG (4731 chunks):
   In this Basic Fantasy RPG, initiative is determined by rolling an initiative number. Each character or creature involved in combat may move up to its encounter movement distance and then attack when its initiative number comes up. Opponents more than 5' apart may move freely, but once within 5' of each other, they are 'engaged' and must follow specific rules.

Additionally, a character using a weapon with a long reach, like a spear, can choose to attack a closing op

The comparison tells a specific story. When retrieval pulls from a larger corpus, it has access to more material, but it also has more opportunities to retrieve the wrong thing. Whether the multi-document answer is better or worse than the single-document answer depends entirely on what landed in those top retrieval slots.

This is not a theoretical concern. It is the central operational challenge of every RAG system that grows beyond a handful of documents. And the only way to catch regressions is to run evaluation every time the corpus changes.

## 6.7 Why Automation Matters (Even in a Lab)
This section answers the unspoken question participants may have been carrying since Section 3: "Why are we doing this so properly in a workshop?"

The answer is practical. Manual workflows hide failure. When you run each step by hand, inspect the output, tweak a parameter, and re-run, you develop an intuition for what works. That intuition is valuable. But it doesn't transfer. It doesn't scale. And it doesn't survive the first time someone else needs to run the pipeline.

The notebook we just walked through is not production infrastructure. But it demonstrates the shape of production infrastructure. Documents go in one end. Queryable, attributed, evaluable context comes out the other. No manual intervention required in the middle.

That matters for three reasons.

First, reproducibility. If a customer asks "why did the answer change?" you need to be able to re-run the pipeline and get the same result. Manual steps introduce variance. Automated steps don't.

Second, onboarding. The customer's team needs to run this without you. If the pipeline requires tribal knowledge ("oh, you have to manually adjust the chunk size for legal documents") it won't survive your departure.

Third, iteration. The first run is expected to be imperfect. The second run should be better for a reason. Automation makes that reason traceable. You changed the chunk size from 1000 to 1500, re-ran the pipeline, re-ran evaluation, and saw that three of five questions improved while one regressed. That's a decision you can explain. That's engineering.

Even partial automation clarifies ownership. Even simple pipelines expose assumptions. And when the customer asks "what happens next month when we add 50 new documents?" you have an answer that isn't "we'll figure it out."

> **Facilitator note**: This is where you quietly align with platform value. You don't need to pitch OpenShift AI or any specific product. The point is that repeatable, automated pipelines are what enterprises need. The connection should be implicit, not stated.

## 6.8 Iteration as a First-Class Concept
The first run is expected to be wrong.

Not catastrophically wrong. Not "something broke" wrong. But wrong in the way that all first attempts at complex systems are wrong: the chunking strategy isn't quite right for every document type, the retrieval returns three chunks when two would be better, the prompt template could be more precise.

The question is never "did it work perfectly the first time?" The question is: "do we know why it behaved the way it did, and can we make a targeted change to improve it?"

This is where evaluation becomes indispensable. Each iteration should answer one question: did this change improve the system? If so, why? If not, where did it fail?

Connect this back to what you just saw in the comparison. The input changed from one document to six. The evaluation caught the differences. Now a decision can be made. That is an iteration. The pipeline didn't change. The data did. And the evaluation framework made the impact visible.

If you can't explain the improvement, you didn't improve. You got lucky. And luck is not a deployment strategy.

## 6.9 Where Model Customization Fits (and Where It Doesn't)

Model customization, whether that's fine-tuning, adapter training, or any form of model adaptation, is a downstream optimization. It assumes that the upstream stages are stable: stable ingestion, stable chunking, stable retrieval, stable evaluation. Without those, tuning amplifies chaos. You're training the model to perform well against a pipeline that's still shifting underneath it.

Most customers will never need model customization. The combination of good ingestion, thoughtful chunking, effective retrieval, and clear prompting will solve the majority of use cases. RAG, done well, is remarkably powerful.

Some customers will absolutely need it. When domain language is implicit rather than explicit, when rules are learned rather than stated, when retrieval gives the model everything it needs and it still fails consistently, that's when model adaptation enters the conversation.

The pipeline tells you which situation you're in. That's the entire point of building it this way. If evaluation shows that the model consistently fails on a particular class of question, and you can confirm that retrieval is returning the right chunks, then you have evidence, not a hunch, that the model itself needs to change.

Without the pipeline, every failure looks like it might be a model problem. With it, you can isolate failures to specific stages and address them precisely.

Fine-tuning is an escalation of effort, not a starting point. The pipeline is what tells you whether that escalation is justified.

## 6.10 What "Good" Looks Like at the End of This Section
Not "a perfectly accurate model." Not "a production-ready system." But:
A pipeline you can reason about. Failures you can explain. Improvements you can attribute to specific changes. Source attribution that makes every answer traceable. An evaluation framework that catches regressions.

And, critically, a story you can retell to a customer. Not a story about tools or frameworks. A story about process. About how you start with documents, make them usable, test whether retrieval is sufficient, and only escalate when the evidence demands it.

The outcome of this lab isn't just an artifact. It's judgment.

## 6.11 The Eval Dataset

**Purpose:**
Capture the current state of the system as a structured, reusable artifact. This becomes the "before" snapshot that later sections pick up.

Everything we've built in this section, the ingestion pipeline, the chunking strategy, the retrieval layer, the model integration, has been working toward a simple question: how good is this system right now, and where exactly does it fall short?

We're going to answer that question concretely. We'll run a curated set of questions through the full pipeline, capture the results, and save them as a JSON file. That file becomes the evidence record. It is what lets us (or anyone who inherits this system) know exactly what worked, what didn't, and why.

This is not a benchmark. It is a diagnostic baseline.

The evaluation set below goes beyond the five questions from Section 6.5. Each question is tagged with a **category** that describes the type of reasoning required: explicit rule lookups, table lookups, terminology interpretation, multi-step rules, and implicit reasoning. Those categories are what let us sort failures into buckets that tell us something actionable about where the system breaks.

The classifier is deliberately simple. It uses keyword heuristics to catch the most obvious failure modes: the model referencing the wrong game system, the model hedging when it has the information, or the model producing an answer that doesn't overlap with what we expected. It is not production-grade. The point is the diagnostic mindset, not the implementation.

In [15]:
eval_dataset = [
    {
        "id": "q01",
        "question": "What happens if a Thief fails an Open Locks attempt?",
        "expected": "The Thief must wait until gaining another level of experience before trying again. It may only be tried once per lock.",
        "category": "explicit_rule"
    },
    {
        "id": "q02",
        "question": "Why can't Elves roll higher than a d6 for hit points?",
        "expected": "Elves use a d6 for hit points because that is the hit die assigned to the Elf combination class in Basic Fantasy RPG.",
        "category": "terminology"
    },
    {
        "id": "q03",
        "question": "Can a character wear leather armor and cast spells?",
        "expected": "Magic-Users and Elves cannot cast spells while wearing armor. Clerics can wear armor and cast spells.",
        "category": "implicit_reasoning"
    },
    {
        "id": "q04",
        "question": "What is the saving throw for a 3rd level Fighter against Dragon Breath?",
        "expected": "Based on the Fighter saving throw table, a 3rd level Fighter has a Dragon Breath saving throw of 15.",
        "category": "table_lookup"
    },
    {
        "id": "q05",
        "question": "How does a Cleric turn undead?",
        "expected": "The Cleric rolls 2d6 and compares the result to the Turn Undead table. Success depends on the Cleric's level and the type of undead.",
        "category": "multi_step_rule"
    },
    {
        "id": "q06",
        "question": "If a character has a Strength of 16, what bonus do they get on melee attack rolls?",
        "expected": "A Strength score of 16 gives a +2 bonus, which applies to melee attack rolls and damage rolls.",
        "category": "table_lookup"
    },
    {
        "id": "q07",
        "question": "What is the difference between a retainer and a hireling?",
        "expected": "Retainers are NPCs who accompany the party on adventures and gain experience. Hirelings are hired for specific non-adventuring tasks.",
        "category": "terminology"
    },
    {
        "id": "q08",
        "question": "When can a Magic-User learn new spells?",
        "expected": "A Magic-User can add spells to their spell book when gaining a level, and may also learn spells found during adventures at the GM's discretion.",
        "category": "implicit_reasoning"
    },
    {
        "id": "q09",
        "question": "What happens to a character at exactly 0 hit points?",
        "expected": "When a character's hit point total reaches 0, the character may be dead. The rules note this may not be the end for the character.",
        "category": "explicit_rule"
    },
    {
        "id": "q10",
        "question": "Can a Halfling use a longbow?",
        "expected": "Halflings may not use Large weapons and must use Medium weapons two-handed. A longbow is a Large weapon, so a Halfling cannot use one.",
        "category": "implicit_reasoning"
    }
]


def classify_result(answer, expected, distances):
    """
    Simple heuristic classification of failure modes.
    Not production-grade. The point is the diagnostic mindset.
    """
    answer_lower = answer.lower()

    # Retrieval failure: top chunk too distant
    if distances and distances[0] > 0.6:
        return "retrieval_failure"

    # Terminology confusion: model references wrong game system
    wrong_system_markers = [
        "dungeons & dragons", "d&d", "5th edition", "5e",
        "pathfinder", "in many rpg", "in most fantasy",
        "typically in role-playing", "in many role-playing",
        "in typical rpg"
    ]
    for marker in wrong_system_markers:
        if marker in answer_lower:
            return "terminology_failure"

    # Hedging when the answer should be present
    hedge_markers = [
        "does not contain enough information",
        "cannot determine",
        "not clear from the context",
        "i'm not sure",
        "the context does not",
        "cannot definitively answer",
        "i cannot definitively"
    ]
    for marker in hedge_markers:
        if marker in answer_lower:
            return "implicit_reasoning_failure"

    # Rough overlap check against expected answer
    stop_words = {
        "the", "a", "an", "is", "are", "of", "to", "and",
        "in", "that", "for", "on", "with", "by", "at", "or"
    }
    expected_terms = set(expected.lower().split()) - stop_words
    answer_terms = set(answer_lower.split())
    overlap = expected_terms & answer_terms

    if len(overlap) < len(expected_terms) * 0.3:
        return "implicit_reasoning_failure"

    return "pass"


# --- Run the evaluation ---

print(f"Running {len(eval_dataset)} questions through the pipeline...\n")
print("=" * 70)

eval_output = []

for item in eval_dataset:
    print(f"  [{item['id']}] {item['question'][:55]}...")

    query_embedding = embed_model.encode(item["question"]).tolist()

    raw_results = collection_multi.query(
        query_embeddings=[query_embedding],
        n_results=3,
        include=["documents", "metadatas", "distances"]
    )

    chunks_returned = raw_results["documents"][0]
    metadatas_returned = raw_results["metadatas"][0]
    distances_returned = raw_results["distances"][0]

    # Build grounded prompt
    context = "\n\n---\n\n".join(chunks_returned)
    system_msg = (
        "You are a rules reference assistant for the Basic Fantasy RPG system. "
        "Answer the question using ONLY the provided context. "
        "If the context does not contain enough information to answer, say so explicitly. "
        "Do not guess. Do not draw from general knowledge of other RPG systems."
    )
    headers_req = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model_id,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {item['question']}"}
        ],
        "max_tokens": 300,
        "temperature": 0
    }

    try:
        response = requests.post(url_chat, headers=headers_req, json=data)
        response.raise_for_status()
        answer = response.json()["choices"][0]["message"]["content"]
    except Exception as e:
        answer = f"[ERROR] {e}"

    classification = classify_result(answer, item["expected"], distances_returned)

    eval_output.append({
        "id": item["id"],
        "question": item["question"],
        "expected": item["expected"],
        "category": item["category"],
        "answer": answer,
        "sources": [m.get("source", "unknown") for m in metadatas_returned],
        "distances": distances_returned,
        "classification": classification
    })

    status = "PASS" if classification == "pass" else classification.replace("_", " ").upper()
    print(f"           -> {status}")

print(f"\n{'=' * 70}")
print(f"Evaluation complete: {len(eval_output)} questions")

counts = {}
for r in eval_output:
    c = r["classification"]
    counts[c] = counts.get(c, 0) + 1

print()
for c in ["pass", "retrieval_failure", "terminology_failure", "implicit_reasoning_failure"]:
    n = counts.get(c, 0)
    pct = n / len(eval_output) * 100
    print(f"  {c.replace('_', ' ').title():35s} {n} ({pct:.0f}%)")

Running 10 questions through the pipeline...

  [q01] What happens if a Thief fails an Open Locks attempt?...
           -> PASS
  [q02] Why can't Elves roll higher than a d6 for hit points?...
           -> IMPLICIT REASONING FAILURE
  [q03] Can a character wear leather armor and cast spells?...
           -> PASS
  [q04] What is the saving throw for a 3rd level Fighter agains...
           -> IMPLICIT REASONING FAILURE
  [q05] How does a Cleric turn undead?...
           -> PASS
  [q06] If a character has a Strength of 16, what bonus do they...
           -> IMPLICIT REASONING FAILURE
  [q07] What is the difference between a retainer and a hirelin...
           -> IMPLICIT REASONING FAILURE
  [q08] When can a Magic-User learn new spells?...
           -> PASS
  [q09] What happens to a character at exactly 0 hit points?...
           -> PASS
  [q10] Can a Halfling use a longbow?...
           -> PASS

Evaluation complete: 10 questions

  Pass                                6 (60%)
  R

> **Facilitator note:** If execution stalled on the previous cell, pre-built results are available in `prebuilt/06/eval_results.json`. Use them without apology. The discussion matters more than the execution.

In [16]:
import os

eval_artifact = {
    "metadata": {
        "source_document": "multi-document corpus",
        "model": model_id,
        "chunking": {
            "strategy": "markdown_aware",
            "chunk_size": 1000,
            "overlap": 200,
            "total_chunks": collection_multi.count(),
            "description": (
                "Splits on markdown headings first, then paragraph breaks, "
                "then word boundaries as a fallback."
            )
        },
        "embedding_model": "ibm-granite/granite-embedding-30m-english",
        "retrieval": {
            "n_results": 3,
            "similarity": "cosine"
        },
        "generated_on": "section_06",
        "note": (
            "Captured in Section 5 using the multi-document RAG pipeline "
            "built in Section 6. This is the baseline for the remaining sections."
        )
    },
    "results": eval_output
}

output_dir = "prebuilt/06"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "eval_results.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(eval_artifact, f, indent=2)

print(f"Saved to {output_path}")
print(f"  {len(eval_output)} results")
print(f"  {collection_multi.count()} total chunks in collection")
print(f"\nThis file is the evaluation handoff artifact.")

Saved to prebuilt/eval_results.json
  10 results
  4731 total chunks in collection

This file is the evaluation handoff artifact.


> **Key Takeawy:**  *"This file is the evidence. Tomorrow, if we decide to go further, it won't be because someone had a hunch. It will be because this evaluation told us to."*

> **Facilitator note:** This file is already included in the lab repository at `prebuilt/06/eval_results.json`. Participants do not need to copy it manually. If they generated their own version here, great. If not, the pre-built copy is waiting for them.